# Spike #191: baseline vs. gradient boosting vs. LSTM

Porovnanie modelov pre hodinovú predikciu výroby (`y = P / kWp`) na **syntetických dátach** zo simulátora #94.
Reálne DKASC dáta nie sú v repe (licencia, ADR #183), takže toto je metodický dry-run, nie produkčný dôkaz.
Výsledky z rovnakého behu sú v `docs/adr/assets/spike-191-results.{json,md}` a zhrnuté v ADR #191.

**Delenie:** expanding-window walk-forward (`2024H1`, `2024H2`, 7-dňová medzera) a leave-one-array-out.
**Metriky:** MAE, RMSE, nMAE (% menovitého výkonu), wMAPE, skill voči fyzikálnemu baseline, čas tréningu/inferencie a veľkosť modelu.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from solarops_ml.forecast import ExperimentConfig, run_experiment
from solarops_ml.forecast.data import DEFAULT_ARRAYS

candidates = [
    Path("../../docs/adr/assets/spike-191-results.json"),
    Path("../docs/adr/assets/spike-191-results.json"),
    Path("docs/adr/assets/spike-191-results.json"),
]
RESULTS_PATH = next((path for path in candidates if path.exists()), candidates[0])

if RESULTS_PATH.exists():
    result = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
else:
    result = run_experiment(
        ExperimentConfig(arrays=DEFAULT_ARRAYS, include_lstm=True, include_loao=True)
    )

print("generated_at:", result["generated_at"])
print("evaluations:", len(result["details"]))
print("arrays:", result["dataset"]["arrays"])

## Súhrn po modeloch

Medián cez fold × pole. `skill` je `1 - MAE_model / MAE_physical`; kladné číslo znamená lepšie ako fyzikálny baseline.


In [ ]:
def summary_table(mode: str) -> pd.DataFrame:
    rows = []
    for model, block in result["summary"][mode].items():
        m = block["metrics"]
        rows.append(
            {
                "model": model,
                "nMAE [%]": m["nmae_pct"]["median"],
                "MAE": m["mae"]["median"],
                "RMSE": m["rmse"]["median"],
                "wMAPE [%]": m["wmape_pct"]["median"],
                "skill vs physical": m["skill_vs_physical"]["median"],
                "train [s]": m["train_seconds"]["median"],
                "inference [ms/1000]": m["inference_ms_per_1000"]["median"],
                "size [B]": m["model_bytes"]["median"],
            }
        )
    order = ["persistence", "physical", "linear", "lightgbm", "lstm"]
    return pd.DataFrame(rows).set_index("model").reindex(order).dropna(how="all")


summary_table("walk_forward").round(3)

In [ ]:
summary_table("leave_one_array_out").round(3)

## Leave-one-array-out po poliach

nMAE [%] pre každé vynechané pole; ukazuje, ako model zovšeobecňuje na inú geometriu/veľkosť poľa.


In [ ]:
details = pd.DataFrame(result["details"])
loao = details[details["mode"] == "leave_one_array_out"]
per_array = (
    loao.groupby(["array", "model"])["nmae_pct"]
    .median()
    .unstack("model")
    .reindex(columns=["persistence", "physical", "linear", "lightgbm", "lstm"])
)
per_array.round(2)

## Odporúčanie

Na syntetických dátach je najlepší **fyzikálny baseline** (nMAE ~0,15 %); LightGBM je vo walk-forward prakticky nerozlíšiteľný (skill +0,037), ale v leave-one-array-out zaostáva (nMAE ~0,40 % vs. 0,15 %) a má ~6 MB model.
LSTM je horšia (~0,29 % WF, ~0,43 % LOAO) a vyžaduje torch. Pre MVP preto ostáva **fyzikálny baseline**; **LightGBM je challenger** a prepne sa len po rerune na reálnych dátach, ak splní gate (≥ 20 % zlepšenie nMAE, nMAE ≤ 6 %). LSTM sa odkladá.


## Obmedzenia

- Syntetické dáta používajú takmer rovnaký fyzikálny model ako baseline, preto je výsledok baselineu z veľkej časti tautologický a poradie modelov sa na reálnych dátach môže zmeniť.
- Všetky polia zdieľajú počasie; LOAO nehodnotí zovšeobecnenie na inú lokalitu ani na reálne poruchy.
- Chýba reálny DKASC dataset a reálne poruchy; výsledky nie sú dôkazom pre produkciu.
- LSTM je malá (1 vrstva, 32 jednotiek, 20 epoch) a neladená; GBM je tiež len rozumný štart, nie výsledok tuningu.
- `temperature_c` z telemetrie sa nepoužíva (obsahuje časť cieľa); používa sa externá `air_temp_c`.
